# 17. Attention Backward and Custom Autograd | Attention 反向传播与自定义 Autograd（Task0 解答版）

材料来源：[datawhalechina/llm-algo-leetcode · 17_Autograd_Basics.ipynb](https://github.com/datawhalechina/llm-algo-leetcode/blob/main/02_PyTorch_Algorithms/17_Autograd_Basics.ipynb)

In [1]:
import torch
import torch.nn.functional as F
import math
print('torch', torch.__version__)

torch 2.9.1+cu128


In [2]:
class CustomAttention(torch.autograd.Function):
    """无 mask、无 dropout 的教学版缩放点积 Attention。

    输入和输出均使用 [B, N, d]；本题只实现单头、等长 Q/K/V 的路径。
    """
    @staticmethod
    def forward(ctx, q, k, v):
        """计算缩放点积 Attention。"""
        if q.ndim != 3 or k.ndim != 3 or v.ndim != 3:
            raise ValueError('q、k、v 必须是 [batch, seq_len, head_dim] 三维张量')
        if q.shape[:2] != k.shape[:2] or k.shape[:2] != v.shape[:2]:
            raise ValueError('q、k、v 的 batch 和序列长度必须一致')
        if q.size(-1) != k.size(-1) or v.size(-1) != q.size(-1):
            raise ValueError('q、k、v 的 head_dim 必须一致')
        if q.device != k.device or k.device != v.device:
            raise ValueError('q、k、v 必须位于同一 device')
        if q.dtype != k.dtype or k.dtype != v.dtype:
            raise TypeError('q、k、v 必须使用相同 dtype')
        d_k = q.size(-1)
        scale = 1.0 / math.sqrt(d_k)
        scores = torch.matmul(q, k.transpose(-2, -1)) * scale
        p = F.softmax(scores, dim=-1)
        out = torch.matmul(p, v)
        # 保存反向传播需要用到的张量：Q/K/V/P
        ctx.save_for_backward(q, k, v, p)
        ctx.scale = scale
        return out

    @staticmethod
    def backward(ctx, dout):
        """根据上游输出梯度返回 q、k、v 的梯度。"""
        q, k, v, p = ctx.saved_tensors
        scale = ctx.scale
        if dout.shape != q.shape:
            raise ValueError('dout 必须与 Attention 输出具有相同形状')
        # TODO 1: dV = P^T @ dO
        dv = torch.matmul(p.transpose(-2, -1), dout)
        # TODO 2: dP = dO @ V^T
        dp = torch.matmul(dout, v.transpose(-2, -1))
        # TODO 3: 穿过 Softmax 求 dS（逐行修正项，避免显式雅可比）
        dp_mul_p = dp * p
        row_sum = dp_mul_p.sum(dim=-1, keepdim=True)
        ds = p * (dp - row_sum)
        # TODO 4: dQ = dS @ K * scale；dK = dS^T @ Q * scale
        dq = torch.matmul(ds, k) * scale
        dk = torch.matmul(ds.transpose(-2, -1), q) * scale
        return dq, dk, dv

In [3]:
def test_attention_backward():
    """验证前向数值、输入梯度形状、梯度数值和输入契约。"""
    torch.manual_seed(42)
    B, N, d = 2, 8, 16
    q = torch.randn(B, N, d, dtype=torch.float64, requires_grad=True)
    k = torch.randn(B, N, d, dtype=torch.float64, requires_grad=True)
    v = torch.randn(B, N, d, dtype=torch.float64, requires_grad=True)

    print("1. 测试前向传播和输出形状...")
    custom_out = CustomAttention.apply(q, k, v)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d)
    ref_out = torch.matmul(F.softmax(scores, dim=-1), v)
    assert custom_out.shape == (B, N, d), "Attention 输出形状不一致！"
    assert torch.isfinite(custom_out).all(), "前向输出包含 NaN 或 Inf！"
    assert torch.allclose(custom_out, ref_out), "前向传播结果不一致！"

    print("\n2. 进行梯度数值检验 (Gradcheck)...")
    assert torch.autograd.gradcheck(CustomAttention.apply, (q, k, v), eps=1e-6, atol=1e-4)

    q_ref, k_ref, v_ref = [t.detach().clone().requires_grad_() for t in (q, k, v)]
    ref_scores = torch.matmul(q_ref, k_ref.transpose(-2, -1)) / math.sqrt(d)
    torch.matmul(F.softmax(ref_scores, dim=-1), v_ref).sum().backward()
    q_custom, k_custom, v_custom = [t.detach().clone().requires_grad_() for t in (q, k, v)]
    CustomAttention.apply(q_custom, k_custom, v_custom).sum().backward()
    for custom_grad, reference_grad, tensor in zip((q_custom.grad, k_custom.grad, v_custom.grad), (q_ref.grad, k_ref.grad, v_ref.grad), (q_custom, k_custom, v_custom)):
        assert custom_grad.shape == tensor.shape, "梯度形状与输入不一致！"
        assert torch.isfinite(custom_grad).all(), "梯度包含 NaN 或 Inf！"
        assert torch.allclose(custom_grad, reference_grad, atol=1e-5), "手写梯度与 PyTorch 结果不一致！"

    bad_k = torch.randn(B, N + 1, d, dtype=torch.float64)
    try:
        CustomAttention.apply(q.detach(), bad_k, v.detach())
    except ValueError as error:
        assert "batch 和序列长度" in str(error)
    else:
        raise AssertionError("不匹配的序列长度应该触发 ValueError")

    print("✅ All Tests Passed! Attention 反向传播实现通过测试。")

test_attention_backward()

1. 测试前向传播和输出形状...

2. 进行梯度数值检验 (Gradcheck)...


✅ All Tests Passed! Attention 反向传播实现通过测试。


## 解答说明（对应 Task0 必做问答 1）

- **为什么 forward 中间结果必须保留到 backward？** `backward()` 是沿前向计算图逆向应用链式法则。反向要计算局部导数（如 `dS = P ⊙ (dP − row_sum(P ⊙ dP))`），局部导数依赖前向的输入/输出（如 Softmax 的输出 P、ReLU 的输入 x）。所以 `ctx.save_for_backward(q, k, v, p)` 保存的对象会驻留到 backward 读完为止。
- **为什么峰值在 backward 附近？** backward 阶段既要读前向驻留的 saved tensors（Q/K/V/P，P 是 B×N×N），又要临时生成 dP、dS 等同量级中间张量，还有参数梯度，叠加达到峰值；backward 结束后 saved tensors 引用释放、计算图销毁，峰值回落。
- **为什么 Attention 反向要保存 Q、K、V 和 P？** dV = Pᵀ·dO 与 dP = dO·Vᵀ 需要 P 和 V；穿过 Softmax 的 dS 需要 P；dQ = dS·K·scale、dK = dSᵀ·Q·scale 需要 K 和 Q。缺一不可。其中 P 形状为 B×N×N，序列变长时显存代价按 N² 放大——这正是 FlashAttention 用 online softmax + 分块重算来避免显式保存 P 的动机。